In [1]:
import random
from collections import Counter

### **Dataset Generation**

In [2]:
# Toy dataset
training_data = [
    [5.1, 3.5, 1.4, 'Setosa'],
    [4.9, 3.0, 1.4, 'Setosa'],
    [7.0, 3.2, 4.7, 'Versicolor'],
    [6.4, 3.2, 4.5, 'Versicolor'],
    [6.3, 3.3, 6.0, 'Virginica'],
    [5.8, 2.7, 5.1, 'Virginica'],
]

# Column labels
header = ["sepal_length", "sepal_width", "petal_length", "label"]

### **Find Unique Values**

In [3]:
def unique_vals(rows, col):
    return set([row[col] for row in rows])

In [4]:
unique_vals(training_data, 3)

{'Setosa', 'Versicolor', 'Virginica'}

### **Class Counts**

In [5]:
# Counts the number of each type of example in a dataset
def class_counts(rows):
    counts = {}
    for row in rows:
        label = row[-1]
        if label not in counts:
            counts[label] = 0
        counts[label] += 1
    return counts

In [6]:
class_counts(training_data)

{'Setosa': 2, 'Versicolor': 2, 'Virginica': 2}

### **Numeric Value**

In [7]:
def is_numeric(value):
    return isinstance(value, int) or isinstance(value, float)

In [8]:
is_numeric(7)

True

In [9]:
is_numeric("sentosa")

False

### **Questions**

In [10]:
# A Question is used to partition a dataset
class Question:
    def __init__(self, column, value, header):
        self.column = column
        self.value = value
        self.header = header

    def match(self, example):
        val = example[self.column]
        if is_numeric(val):
            return val >= self.value
        else:
            return val == self.value

    def __repr__(self):
        condition = "=="
        if is_numeric(self.value):
            condition = ">="
        return "Is %s %s %s?" % (self.header[self.column], condition, str(self.value))


In [11]:
q = Question(0, 6.0, header)
example = training_data[0]
q.match(example)

False

In [12]:
q

Is sepal_length >= 6.0?

### **Partition**

In [13]:
def partition(rows, question):
    true_rows, false_rows = [], []
    for row in rows:
        if question.match(row):
            true_rows.append(row)
        else:
            false_rows.append(row)
    return true_rows, false_rows

In [14]:
true_rows, false_rows = partition(training_data, Question(2, 5.0, header))

In [15]:
true_rows

[[6.3, 3.3, 6.0, 'Virginica'], [5.8, 2.7, 5.1, 'Virginica']]

In [16]:
false_rows

[[5.1, 3.5, 1.4, 'Setosa'],
 [4.9, 3.0, 1.4, 'Setosa'],
 [7.0, 3.2, 4.7, 'Versicolor'],
 [6.4, 3.2, 4.5, 'Versicolor']]

### **Gini Impurity**

In [17]:
def gini(rows):
    counts = class_counts(rows)
    impurity = 1
    for lbl in counts:
        prob_of_lbl = counts[lbl] / float(len(rows))
        impurity -= prob_of_lbl**2
    return impurity

In [19]:
no_mixing = [['Apple'], ['Apple']]
gini(no_mixing) # returns 0

0.0

In [20]:
some_mixing = [['Apple'], ['Apple'], ['Grape'], ['Berry']]
gini(some_mixing)

0.625

In [21]:
high_mixing = [['Apple'], ['Apple'], ['Grape'], ['Berry'], ['Grape'], ['Banana'], ['Stawberry'], ['Grape'], ['Berry']]
gini(high_mixing)

0.765432098765432

### **Information Gain**

In [22]:
def info_gain(left, right, current_uncertainty):
    p = float(len(left)) / (len(left) + len(right))
    return current_uncertainty - p * gini(left) - (1 - p) * gini(right)

In [23]:
current_uncertainty = gini(training_data)
true_rows, false_rows = partition(training_data, Question(0, 6.0, header))
info_gain(true_rows, false_rows, current_uncertainty)

0.22222222222222207

### **Find Best Split**

In [24]:
def find_best_split(rows, features_to_consider):
    best_gain = 0
    best_question = None
    current_uncertainty = gini(rows)

    # We only iterate over a random subset of features
    for col in features_to_consider:
        values = set([row[col] for row in rows])
        for val in values:
            question = Question(col, val, header)
            true_rows, false_rows = partition(rows, question)
            if len(true_rows) == 0 or len(false_rows) == 0:
                continue
            gain = info_gain(true_rows, false_rows, current_uncertainty)
            if gain >= best_gain:
                best_gain, best_question = gain, question
    return best_gain, best_question


In [25]:
all_features = list(range(len(training_data[0]) - 1))
find_best_split(training_data, all_features)

(0.3333333333333332, Is petal_length >= 4.5?)

### **Leaf & Decision Node Class**

In [26]:
# A Leaf node classifies data
class Leaf:
    def __init__(self, rows):
        self.predictions = class_counts(rows)

# A Decision Node asks a question
class Decision_Node:
    def __init__(self, question, true_branch, false_branch):
        self.question = question
        self.true_branch = true_branch
        self.false_branch = false_branch

### **Build Tree**

In [27]:
def build_tree(rows, features):
    gain, question = find_best_split(rows, features)
    if gain == 0:
        return Leaf(rows)
    true_rows, false_rows = partition(rows, question)
    true_branch = build_tree(true_rows, features)
    false_branch = build_tree(false_rows, features)
    return Decision_Node(question, true_branch, false_branch)

### **Print Tree**

In [28]:
def print_tree(node, spacing=""):
    if isinstance(node, Leaf):
        print(spacing + "Predict", node.predictions)
        return
    print(spacing + str(node.question))
    print(spacing + '--> True:')
    print_tree(node.true_branch, spacing + "  ")
    print(spacing + '--> False:')
    print_tree(node.false_branch, spacing + "  ")

### **Classify & Print Leaf**

In [29]:
def classify(row, node):
    if isinstance(node, Leaf):
        return node.predictions
    if node.question.match(row):
        return classify(row, node.true_branch)
    else:
        return classify(row, node.false_branch)

In [30]:
def print_leaf(counts):
    total = sum(counts.values()) * 1.0
    probs = {}
    for lbl in counts.keys():
        probs[lbl] = str(int(counts[lbl] / total * 100)) + "%"
    return probs

### **Prediction & Bootstrap**

In [31]:
def get_prediction(predictions):
    combined_counts = Counter()
    for pred in predictions:
        combined_counts.update(pred)
    # Return the label with the highest count
    return combined_counts.most_common(1)[0][0]

In [32]:
def bootstrap_sample(data):
    sample = []
    n_samples = len(data)
    for _ in range(n_samples):
        sample.append(random.choice(data))
    return sample

### **Build Random Forest**

In [33]:
def build_random_forest(data, n_trees, n_features):
    forest = []
    all_features = list(range(len(data[0]) - 1))
    for i in range(n_trees):
        # Bootstrap sample the data
        sample = bootstrap_sample(data)
        # Randomly select a subset of features
        features = random.sample(all_features, n_features)
        # Build a decision tree on the sample and features
        tree = build_tree(sample, features)
        forest.append(tree)
    return forest

### **Making Random Forest**

In [34]:
def random_forest_predict(forest, row):
    predictions = [classify(row, tree) for tree in forest]
    return get_prediction(predictions)

In [35]:
# Build a Random Forest with 5 trees and 2 features per split
num_trees = 5
num_features = 2
my_forest = build_random_forest(training_data, num_trees, num_features)

In [36]:
# Evaluate the forest
testing_data = [
        [5.0, 3.4, 1.5, 'Setosa'],
        [6.8, 3.0, 4.8, 'Versicolor'],
        [6.5, 3.1, 5.2, 'Virginica'],
        [4.8, 3.4, 1.6, 'Setosa'],
    ]

In [37]:
print("Random Forest Predictions:")
for row in testing_data:
  actual_label = row[-1]
  predicted_label = random_forest_predict(my_forest, row)
  print("Actual: %s. Predicted: %s" % (actual_label, predicted_label))

Random Forest Predictions:
Actual: Setosa. Predicted: Setosa
Actual: Versicolor. Predicted: Versicolor
Actual: Virginica. Predicted: Virginica
Actual: Setosa. Predicted: Setosa
